# ANIMA — Dataset Quality Check (53-EDO)

A sincere, end-to-end QC of the 53-TET MPE dataset. For each check we print a verdict — **OK** or **FAIL** — so we can glance at the notebook and know if anything is rotten.

We answer these questions:

1. **Inventory** — do all 14 transformation directories exist, and do they contain the same songs (i.e. are the 14 types really *parallel* views of the same corpus)?
2. **Parallel canary** — pick canonical songs and load the same song across all 14 types. Their mod-53 pitch-class sets should be *different* (otherwise the transformations are fake). `type_0_major` should be 12-TET-aligned (pcs ⊂ the 12 pythagorean anchors), while the other 13 should contain non-12-TET steps.
3. **Microtonality metric** — on a random sample in each type, what fraction of pitches is within ±7¢ of 12-TET? `type_0_major` should be ~100%; microtonal types should be notably below 100%.
4. **Tokenizer round-trip** — `parse_mpe_midi → encode → decode → chords` preserves pitches.
5. **Preprocess round-trip** — the full `preprocess_song` pipeline preserves pitches.
6. **A/B listen** — pipeline-output vs source for one song per type.
7. **Export WAVs** — for manual inspection in `dataset/audio/qc/`.

> 53-EDO recap: 1 octave = 53 equal steps of ≈22.64 ¢. 12-TET semitones map (by minimum-error rounding) to 53-TET steps {0, 4, 9, 13, 18, 22, 27, 31, 35, 40, 44, 49}, with max deviation ≈7 ¢. Any 53-TET pitch class *not* in that set is genuinely microtonal.


In [3]:
import os, sys, json, random, importlib, collections
from pathlib import Path
from IPython.display import Audio, display

import numpy as np
import mido

SRC_DIR = Path('/home/david/Projects/ANIMA_Microtonal_GPT/src')
ROOT_DIR = SRC_DIR.parent
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import tokenizer as tok_mod
import generate as gen
try:
    import preprocess as preproc
    HAVE_PREPROC = True
except Exception as e:
    print(f'⚠️  preprocess import failed ({type(e).__name__}: {e}) — check 5 will be skipped.')
    HAVE_PREPROC = False

tokenizer = tok_mod.MPETokenizer()

MIDI_ROOT = ROOT_DIR / 'dataset' / 'midi_files' / '53_tet_mpe'
QC_OUTPUT = ROOT_DIR / 'dataset' / 'generated' / 'qc'
WAV_OUTPUT = ROOT_DIR / 'dataset' / 'audio' / 'qc'
QC_OUTPUT.mkdir(parents=True, exist_ok=True)
WAV_OUTPUT.mkdir(parents=True, exist_ok=True)

# --- 53-EDO helpers ---------------------------------------------------------
TET = 53
# 12-TET semitones rounded to nearest 53-TET step (minimum-error mapping).
# Max deviation from exact 12-TET: ~7 cents.
TWELVE_TET_ANCHORS_53 = {round(i * TET / 12) % TET for i in range(12)}
assert len(TWELVE_TET_ANCHORS_53) == 12, TWELVE_TET_ANCHORS_53
CENTS_PER_STEP = 1200.0 / TET  # ≈ 22.6415 ¢

def pc53(step):
    """Return pitch class modulo 53."""
    return step % TET

def cents_from_nearest_12tet(step):
    """Signed cents between a 53-TET step and its nearest 12-TET semitone."""
    c_abs = step * CENTS_PER_STEP
    nearest_semitone = round(c_abs / 100.0)
    return c_abs - nearest_semitone * 100.0

def is_12tet_anchor(step, tol_cents=7.0):
    return abs(cents_from_nearest_12tet(step)) <= tol_cents

print(f'Root dir        : {ROOT_DIR}')
print(f'MIDI_ROOT       : {MIDI_ROOT}')
print(f'Vocab size      : {tokenizer.vocab_size}')
print(f'12-TET anchors  : {sorted(TWELVE_TET_ANCHORS_53)}')
print(f'Cents/step      : {CENTS_PER_STEP:.4f}')


⚠️  preprocess import failed (ImportError: cannot import name 'DEFAULT_DELTA' from 'eigenspace' (/home/david/Projects/ANIMA_Microtonal_GPT/src/eigenspace.py)) — check 5 will be skipped.
Root dir        : /home/david/Projects/ANIMA_Microtonal_GPT
MIDI_ROOT       : /home/david/Projects/ANIMA_Microtonal_GPT/dataset/midi_files/53_tet_mpe
Vocab size      : 2930
12-TET anchors  : [0, 4, 9, 13, 18, 22, 26, 31, 35, 40, 44, 49]
Cents/step      : 22.6415


## Check 1 — Inventory

All 14 type directories must exist and contain roughly the same number of songs (they are parallel transformations of the same corpus). We also verify that each type contains the same set of song-IDs as `type_0_major` (the reference).


In [4]:
EXPECTED_TYPES = [
    'type_0_major', 'type_0_minor',
    'type_1_minor', 'type_1_neutral',
    'type_2_minor', 'type_2_subminor',
    'type_3_major', 'type_3_minor',
    'type_4_minor', 'type_4_upmajor',
    'type_5_major_v2', 'type_5_minor',
    'type_6_minor', 'type_6_neutral_n',
]

type_dirs = {t: MIDI_ROOT / t for t in EXPECTED_TYPES}
missing_dirs = [t for t, d in type_dirs.items() if not d.is_dir()]
assert not missing_dirs, f'Missing type dirs: {missing_dirs}'

# Song-id = leading numeric prefix of the filename ('12345_...').
def song_id(p):
    stem = p.stem
    return stem.split('_', 1)[0] if stem.split('_', 1)[0].isdigit() else stem

counts = {}
id_sets = {}
for t, d in type_dirs.items():
    mids = list(d.glob('*.mid'))
    counts[t] = len(mids)
    id_sets[t] = {song_id(p) for p in mids}

ref = 'type_0_major'
ref_ids = id_sets[ref]
print(f'{"type":20s}  {"n_files":>8s}  missing_vs_ref  extra_vs_ref')
all_ok = True
for t in EXPECTED_TYPES:
    miss = len(ref_ids - id_sets[t])
    extra = len(id_sets[t] - ref_ids)
    flag = '' if miss == 0 and extra == 0 else '  <-- DRIFT'
    if miss or extra:
        all_ok = False
    print(f'{t:20s}  {counts[t]:>8d}  {miss:>14d}  {extra:>12d}{flag}')
print()
print('VERDICT:', 'OK — all 14 types parallel' if all_ok else 'FAIL — song-ID drift between types')


type                   n_files  missing_vs_ref  extra_vs_ref
type_0_major             48060               0             0
type_0_minor             48060               0             0
type_1_minor             48060               0             0
type_1_neutral           48060               0             0
type_2_minor             48060               0             0
type_2_subminor          48060               0             0
type_3_major             48060               0             0
type_3_minor             48060               0             0
type_4_minor             48060               0             0
type_4_upmajor           48060               0             0
type_5_major_v2          48060               0             0
type_5_minor             48060               0             0
type_6_minor             48060               0             0
type_6_neutral_n         48060               0             0

VERDICT: OK — all 14 types parallel


## Check 2 — Parallel canary

Pick a few canonical songs that are present in every type, parse one MIDI per type, and compare the set of **mod-53 pitch classes** used. Two things must hold:

1. `type_0_major` has pitch classes ⊂ the 12-TET anchor set (≤ ~7¢ deviation everywhere).
2. At least some other types contain pitch classes *outside* the 12-TET anchor set — i.e. genuine microtonal content.

If every type produced the same pitch-class signature, the transformations would be fake.


In [5]:
CANARIES = ['Autumn Leaves', 'Blue Bossa', 'Misty', 'Giant Steps']

def find_canary(type_dir, fragment):
    hits = sorted(p for p in type_dir.iterdir()
                  if p.suffix == '.mid' and f'_{fragment}_C_' in p.name)
    hits.sort(key=lambda p: (len(p.name), p.name))
    return hits[0] if hits else None

micro_type_count = collections.Counter()

for song in CANARIES:
    print(f'\n── {song} ' + '─'*(70-len(song)))
    per_type = {}
    for t in EXPECTED_TYPES:
        p = find_canary(type_dirs[t], song)
        if p is None:
            per_type[t] = None
            continue
        chords = tok_mod.parse_mpe_midi(str(p))
        pcs = sorted({pc53(n['step_53']) for c in chords for n in c['notes']})
        per_type[t] = pcs

    # How many distinct pc-signatures across types?
    sigs = {t: tuple(v) for t, v in per_type.items() if v is not None}
    distinct = len(set(sigs.values()))

    for t in EXPECTED_TYPES:
        pcs = per_type[t]
        if pcs is None:
            print(f'  {t:20s}  (missing)')
            continue
        non_12tet = [p for p in pcs if p not in TWELVE_TET_ANCHORS_53]
        tag = '12-TET-aligned' if not non_12tet else f'has {len(non_12tet)} non-12-TET pc(s): {non_12tet}'
        if non_12tet:
            micro_type_count[t] += 1
        print(f'  {t:20s}  {len(pcs):2d} pcs  {tag}')

    print(f'  -> {distinct} distinct pitch-class signatures across {len(sigs)} types')

print()
print('Microtonal (non-12-TET pc present) count across canaries, per type:')
for t in EXPECTED_TYPES:
    print(f'  {t:20s}  {micro_type_count[t]}/{len(CANARIES)}')

ok = (micro_type_count['type_0_major'] == 0
      and sum(1 for t in EXPECTED_TYPES if t != 'type_0_major' and micro_type_count[t] > 0) >= 10)
print()
print('VERDICT:', 'OK — transformations are genuinely distinct' if ok else 'SUSPECT — inspect above')



── Autumn Leaves ─────────────────────────────────────────────────────────
  type_0_major          12 pcs  has 2 non-12-TET pc(s): [14, 36]
  type_0_minor          12 pcs  has 1 non-12-TET pc(s): [48]
  type_1_minor          12 pcs  has 5 non-12-TET pc(s): [7, 14, 29, 36, 45]
  type_1_neutral        12 pcs  has 5 non-12-TET pc(s): [8, 15, 39, 46, 50]
  type_2_minor          12 pcs  has 10 non-12-TET pc(s): [5, 10, 14, 19, 24, 28, 36, 41, 46, 50]
  type_2_subminor       12 pcs  has 5 non-12-TET pc(s): [12, 17, 34, 39, 48]
  type_3_major          12 pcs  has 3 non-12-TET pc(s): [17, 39, 48]
  type_3_minor          12 pcs  has 7 non-12-TET pc(s): [5, 10, 14, 23, 27, 36, 45]
  type_4_minor          12 pcs  has 6 non-12-TET pc(s): [12, 17, 34, 38, 43, 48]
  type_4_upmajor        12 pcs  has 8 non-12-TET pc(s): [5, 10, 14, 19, 36, 41, 46, 50]
  type_5_major_v2       12 pcs  has 5 non-12-TET pc(s): [14, 23, 27, 39, 48]
  type_5_minor          12 pcs  has 7 non-12-TET pc(s): [14, 23, 28, 32, 

## Check 3 — Microtonality metric per type

Random sample of songs per type; measure the fraction of pitches whose 53-TET step falls **within ±7¢** of a 12-TET semitone. Expected:

- `type_0_major` ≈ **100%** (it IS 12-TET by construction)
- all other types noticeably **< 100%** (they use genuine commas)

A type that matches `type_0_major`'s profile would be a fake transformation.


In [6]:
random.seed(7)
SAMPLE_PER_TYPE = 30
TOL_CENTS = 7.0

print(f'Sampling {SAMPLE_PER_TYPE} random songs per type, tolerance ±{TOL_CENTS}¢')
print(f'{"type":20s}  {"n_songs":>7s}  {"n_notes":>9s}  {"%12-TET":>8s}  {"mean|dev|¢":>11s}  verdict')

results = {}
for t in EXPECTED_TYPES:
    mids = list(type_dirs[t].glob('*.mid'))
    sample = random.sample(mids, min(SAMPLE_PER_TYPE, len(mids)))
    all_steps = []
    for p in sample:
        try:
            chords = tok_mod.parse_mpe_midi(str(p))
            for c in chords:
                for n in c['notes']:
                    all_steps.append(n['step_53'])
        except Exception:
            pass
    devs = np.array([abs(cents_from_nearest_12tet(s)) for s in all_steps])
    pct_12 = 100.0 * np.mean(devs <= TOL_CENTS) if len(devs) else 0.0
    mean_dev = float(np.mean(devs)) if len(devs) else 0.0
    results[t] = (pct_12, mean_dev, len(devs))

# Baseline expectation: type_0_major ~ 100%.
t0 = results['type_0_major'][0]
verdicts = {}
for t, (pct, mean_dev, n) in results.items():
    if t == 'type_0_major':
        verdicts[t] = 'OK' if pct > 99.0 else 'FAIL'
    else:
        # Microtonal types should be at least 2 percentage points below type_0_major,
        # OR have mean|dev| notably above 1¢.
        verdicts[t] = 'OK' if (t0 - pct) > 2.0 or mean_dev > 2.0 else 'SUSPECT'

for t in EXPECTED_TYPES:
    pct, mean_dev, n = results[t]
    print(f'{t:20s}  {SAMPLE_PER_TYPE:>7d}  {n:>9d}  {pct:>7.2f}%  {mean_dev:>10.3f}   {verdicts[t]}')

bad = [t for t, v in verdicts.items() if v != 'OK']
print()
print('VERDICT:', 'OK — all 14 types behave as expected' if not bad else f'REVIEW: {bad}')


Sampling 30 random songs per type, tolerance ±7.0¢
type                  n_songs    n_notes   %12-TET   mean|dev|¢  verdict
type_0_major               30      18635    48.69%       7.497   FAIL
type_0_minor               30      21451    50.83%       7.310   OK
type_1_minor               30      18600    33.48%      17.612   OK
type_1_neutral             30      21258    33.47%      13.522   OK
type_2_minor               30      15882    18.49%      20.361   OK
type_2_subminor            30      18807    45.27%       9.605   OK
type_3_major               30      19182    38.25%       9.471   OK
type_3_minor               30      20462    37.99%      10.967   OK
type_4_minor               30      17803    42.64%      13.809   OK
type_4_upmajor             30      17717    31.45%      16.123   OK
type_5_major_v2            30      16607    41.30%       9.679   OK
type_5_minor               30      17773    22.56%      15.834   OK
type_6_minor               30      18731    15.47%      17

## Check 4 — Tokenizer round-trip

`parse_mpe_midi → encode → decode → extract_chords_summary` must preserve the sorted pitch list of every chord. We compare the first K chords.


In [7]:
random.seed(11)
K_CHORDS = 8
per_type_results = {}

def chord_pitches(chord):
    return sorted(n['step_53'] for n in chord['notes'])

for t in EXPECTED_TYPES:
    p = random.choice(list(type_dirs[t].glob('*.mid')))
    src_chords = tok_mod.parse_mpe_midi(str(p))
    # Encode -> decode round-trip.
    try:
        token_ids = tokenizer.encode(src_chords)
    except Exception as e:
        per_type_results[t] = (p.name, 'ENCODE_ERR', str(e))
        continue
    decoded_tokens = [tokenizer.id_to_token.get(i, '?') for i in token_ids]
    pipe_chords = gen.extract_chords_summary(decoded_tokens)

    match = 0
    total = min(K_CHORDS, len(src_chords), len(pipe_chords))
    for i in range(total):
        s = chord_pitches(src_chords[i])
        pp = sorted(pipe_chords[i]['pitches'])
        if s == pp:
            match += 1
    per_type_results[t] = (p.name, match, total)

print(f'{"type":20s}  {"match":>6s}  file')
all_ok = True
for t in EXPECTED_TYPES:
    fn, m, total = per_type_results[t]
    flag = 'OK' if isinstance(m, int) and m == total and total > 0 else 'FAIL'
    if flag == 'FAIL':
        all_ok = False
    score = f'{m}/{total}' if isinstance(m, int) else f'ERR'
    print(f'{t:20s}  {score:>6s}  {flag}   {fn}')
print()
print('VERDICT:', 'OK — tokenizer round-trip lossless' if all_ok else 'FAIL — see rows above')


type                   match  file
type_0_major             ERR  FAIL   15645_Sorriso Aberto_A_minor_type_0_major.mid
type_0_minor             ERR  FAIL   36676_Tickle-Toe_E_major_type_0_minor.mid
type_1_minor             ERR  FAIL   15486_Zip-A-Dee-Doo-Dah Song of the South_Gb_major_type_1_minor.mid
type_1_neutral           ERR  FAIL   20155_Rocket Love_G_minor_type_1_neutral.mid
type_2_minor             ERR  FAIL   22948_I Love Paris_E_minor_type_2_minor.mid
type_2_subminor          ERR  FAIL   24836_Diverse_Ab_minor_type_2_subminor.mid
type_3_major             ERR  FAIL   42587_Christmas Swing_B_major_type_3_major.mid
type_3_minor             ERR  FAIL   12115_Fim da Tristeza_G_major_type_3_minor.mid
type_4_minor             ERR  FAIL   39403_So What_G_minor_type_4_minor.mid
type_4_upmajor           ERR  FAIL   29235_Stolen Moments_Eb_minor_type_4_upmajor.mid
type_5_major_v2          ERR  FAIL   09701_Woodstock_F_minor_type_5_major_v2.mid
type_5_minor             ERR  FAIL   14039_B

## Check 5 — Preprocess round-trip

Full pipeline (`preprocess_song`) parse → encode → returned token_ids → decode → chord pitches, compared to the source MIDI's pitches. Skipped gracefully if `preprocess` module failed to import above.


In [8]:
if not HAVE_PREPROC:
    print('skipped — preprocess module not importable')
else:
    random.seed(13)
    vocab_path = ROOT_DIR / 'dataset' / 'tokenized' / 'vocab.json'
    vocab_data = json.load(open(vocab_path))
    id2tok = {v: k for k, v in vocab_data['token_to_id'].items()}

    sample_types = EXPECTED_TYPES[:6]  # 6 types — enough for a smoke-test
    total_ok = 0
    total_checked = 0
    for t in sample_types:
        p = random.choice(list(type_dirs[t].glob('*.mid')))
        src_chords = tok_mod.parse_mpe_midi(str(p))

        res = preproc.preprocess_song(str(p), tokenizer=tokenizer)
        if res is None:
            print(f'{t:20s}  SKIP (preprocess returned None)  {p.name}')
            continue
        decoded = [id2tok.get(i, '?') for i in res['token_ids']]
        pipe_chords = gen.extract_chords_summary(decoded)
        k = min(8, len(src_chords), len(pipe_chords))
        hits = sum(
            1 for i in range(k)
            if sorted(n['step_53'] for n in src_chords[i]['notes']) == sorted(pipe_chords[i]['pitches'])
        )
        total_ok += hits
        total_checked += k
        tag = 'OK' if hits == k else 'FAIL'
        print(f'{t:20s}  {hits}/{k}  {tag}   {p.name}')

    print()
    print(f'VERDICT:', 'OK' if total_ok == total_checked else f'{total_ok}/{total_checked} chords matched')


skipped — preprocess module not importable


## Check 6 — A/B listen

For a handful of types, render the pipeline output vs source MIDI so we can *hear* whether anything mangled. Sine waves (53-EDO preserving).


In [ ]:
import play_mpe as pm
importlib.reload(pm)

PLAYBACK_SPEED = 1.2
WAVEFORM       = 'sine'
REVERB         = 20

listen_types = ['type_0_major', 'type_1_neutral', 'type_2_subminor', 'type_4_upmajor', 'type_6_neutral_n']
random.seed(17)
listen_files = []
for t in listen_types:
    p = random.choice(list(type_dirs[t].glob('*.mid')))
    # Pipeline: parse -> tokenize -> decode -> chords_to_midi -> render
    src_chords = tok_mod.parse_mpe_midi(str(p))
    out_path = QC_OUTPUT / f'qc_{t}.mid'
    tokenizer.chords_to_midi(src_chords, out_path, tpb=960, tempo_bpm=120)
    listen_files.append((t, p, out_path))

for t, src, out in listen_files:
    print(f'\n── {t} ── {src.name}')
    print('▶ Pipeline output:')
    a, sr = pm.render_mpe_to_audio_data(str(out), speed=PLAYBACK_SPEED, waveform=WAVEFORM, reverb=REVERB)
    if a is not None: display(Audio(a, rate=sr))
    print('▶ Source MIDI:')
    a, sr = pm.render_mpe_to_audio_data(str(src), speed=PLAYBACK_SPEED, waveform=WAVEFORM, reverb=REVERB)
    if a is not None: display(Audio(a, rate=sr))



── type_0_major ── 46322_Eronel_D_major_type_0_major.mid
▶ Pipeline output:
play_mpe | waveform: sine, reverb: 20%
Loading qc_type_0_major.mid...
Rendering 657 notes. Total duration: 260.90s (Speed: 1.2x)


## Check 7 — Export WAVs

Save the QC pipeline-output WAVs under `dataset/audio/qc/` for offline inspection.


In [ ]:
for t, src, out in listen_files:
    wav_out = WAV_OUTPUT / (out.stem + '.wav')
    pm.render_mpe_to_audio_data(
        str(out), speed=PLAYBACK_SPEED, waveform=WAVEFORM, reverb=REVERB,
        save_path=str(wav_out),
    )
    print('saved:', wav_out.relative_to(ROOT_DIR))
print('Done.')
